In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pd.read_csv("../data/raw/aqi_hourly_v2.csv", index_col=0, parse_dates=True)

# chhote gaps bhar diye
df["AQI"] = df["AQI"].interpolate(limit=6, limit_area="inside")

# time aur wind direction circular hote hain, isliye sin cos
df["hour_sin"] = np.sin(2 * np.pi * df.index.hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * df.index.hour / 24)
df["month_sin"] = np.sin(2 * np.pi * df.index.month / 12)
df["month_cos"] = np.cos(2 * np.pi * df.index.month / 12)
df["wd_sin"] = np.sin(np.deg2rad(df["wind_direction_10m"]))
df["wd_cos"] = np.cos(np.deg2rad(df["wind_direction_10m"]))

feat_cols = ["AQI", "temperature_2m", "relative_humidity_2m", "wind_speed_10m",
             "wd_sin", "wd_cos", "surface_pressure", "precipitation",
             "hour_sin", "hour_cos", "month_sin", "month_cos"]
print(df[feat_cols].shape)

(74544, 12)


In [3]:
train_rows = df.index.year < 2024
mu = df.loc[train_rows, feat_cols].mean()
sd = df.loc[train_rows, feat_cols].std()

scaled = ((df[feat_cols] - mu) / sd).values.astype("float32")
aqi_arr = df["AQI"].values

In [4]:
W = 72
H = np.array([6, 12, 24, 48, 72])
N = len(df)
nan_aqi = np.isnan(aqi_arr)

# wahi windows jinme pichhle 72 ghante aur saare targets mein nan na ho
idx = []
for t in range(W - 1, N - H.max()):
    if nan_aqi[t - W + 1 : t + 1].any() or nan_aqi[t + H].any():
        continue
    idx.append(t)
idx = np.array(idx)

X = np.stack([scaled[t - W + 1 : t + 1] for t in idx])
Y = (aqi_arr[idx[:, None] + H] / 500).astype("float32")
years = df.index[idx].year
print(X.shape, Y.shape)

(42927, 72, 12) (42927, 5)


In [5]:
tr = years < 2024
va = years == 2024
te = years == 2025

X_tr, Y_tr = X[tr], Y[tr]
X_va, Y_va = X[va], Y[va]
X_te, Y_te = X[te], Y[te]
print(X_tr.shape, X_va.shape, X_te.shape)

(27580, 72, 12) (8672, 72, 12) (6675, 72, 12)


In [6]:
class LSTMNet(nn.Module):
    def __init__(self, n_feat, hidden=64, n_out=5):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden, n_out)

    def forward(self, x):
        out, _ = self.lstm(x)
        # sirf akhri time step ka output use kiya
        return self.fc(out[:, -1])

In [7]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(device)

model = LSTMNet(X.shape[2]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.L1Loss()

Xtr_t, Ytr_t = torch.tensor(X_tr), torch.tensor(Y_tr)

def predict(Xnp):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(Xnp), 512):
            xb = torch.tensor(Xnp[i:i + 512]).to(device)
            out.append(model(xb).cpu().numpy())
    return np.concatenate(out)

best, bad, best_state = 1e9, 0, None
for ep in range(40):
    model.train()
    perm = torch.randperm(len(Xtr_t))
    for i in range(0, len(perm), 256):
        b = perm[i:i + 256]
        xb, yb = Xtr_t[b].to(device), Ytr_t[b].to(device)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()

    # validation mae aqi ke units mein
    val_mae = np.abs(predict(X_va) - Y_va).mean() * 500
    print(ep, round(val_mae, 2))

    if val_mae < best:
        best, bad = val_mae, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad += 1
        if bad >= 5:
            break

model.load_state_dict(best_state)

mps
0 49.9
1 49.06
2 47.88
3 47.7
4 47.57
5 50.54
6 48.13
7 48.53
8 48.96
9 49.23


<All keys matched successfully>

In [8]:
pred = np.clip(predict(X_te) * 500, 0, 500)
true = Y_te * 500
now = aqi_arr[idx[te]]   # window ke akhri ghante ka aqi = persistence

# xgboost ke numbers pichhle notebook ki cell 7 se
xgb_mae = [40.6, 43.6, 45.9, 50.5, 54.6]
xgb_rmse = [55.7, 60.5, 63.2, 68.2, 72.8]

rows = []
for j, h in enumerate(H):
    rows.append({"horizon": h,
                 "lstm_mae": mean_absolute_error(true[:, j], pred[:, j]),
                 "xgb_mae": xgb_mae[j],
                 "pers_mae": mean_absolute_error(true[:, j], now),
                 "lstm_rmse": np.sqrt(mean_squared_error(true[:, j], pred[:, j])),
                 "xgb_rmse": xgb_rmse[j]})
print(pd.DataFrame(rows).round(1))

   horizon  lstm_mae  xgb_mae  pers_mae  lstm_rmse  xgb_rmse
0        6      45.1     40.6      53.3       61.6      55.7
1       12      47.0     43.6      61.8       64.2      60.5
2       24      45.0     45.9      52.1       63.4      63.2
3       48      49.5     50.5      62.7       68.2      68.2
4       72      53.4     54.6      67.1       73.3      72.8


In [9]:
import os
os.makedirs("models", exist_ok=True)
torch.save(best_state, "models/lstm_aqi.pt")